# generate complete exemples with GPT5.4
read each example and the abrege, and produce all possible examples (no more # or options)

In [25]:
import pandas as pd
import json
df_completed_templates_sein= pd.read_json("data/df_templatesCompletedTitleMacro.json")
df_completed_templates_sein

,Code,Abrégé,Type,ADICAP,Text,sampling_mode,organ,pathology_type,complete_text,type_diagnostique
0,FMGSAFB,FIBROADENOME : BIOPSIE,X,OHGSA0P1,Examen microscopique\nOn observe une lésion fi...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),FIBRO-ADENOME TUBULEUX SIMPLE,SEIN GAUCHE : MICROBIOPSIES D'UN NODULE ACR4 d...,biopsie
1,FMGSAFBS,FIBROADENOMES : BIOPSIES,X,OHGSA0P1,Examen microscopique\nOn observe des lésions f...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),FIBRO-ADENOME TUBULEUX SIMPLE,MICROBIOPSIE MAMMAIRE 14G DU QSI DU SEIN GAUCH...,biopsie
2,FMGSAFM,FIBROADENOME MYXOIDE : BIOPSIE ou EXERESE,X,OHGSA0P1,Examen microscopique\nOn observe une lésion fi...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),FIBRO-ADENOME TUBULEUX SIMPLE,SEIN GAUCHE : MICROBIOPSIES ECHOGUIDEES D’UNE ...,biopsie
3,FMGSAFP,FIBROADENOME : PO,X,OHGSA0P1,Examen microscopique\nOn observe une lésion fi...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),FIBRO-ADENOME TUBULEUX SIMPLE,TUMORECTOMIE DU SEIN GAUCHE\n\nExamen macrosco...,tumorectomie
4,FMGSAFPS,FIBROADENOMES : PO,X,OHGSA0P1,Examen microscopique\nOn observe des lésions f...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),FIBRO-ADENOME TUBULEUX SIMPLE,TUMORECTOMIE DU SEIN GAUCHE\n\nExamen macrosco...,tumorectomie
...,...,...,...,...,...,...,...,...,...,...
323,GSKCOL11,P_K.CANAL. MUCINEUX,X,OHGSA7M4,Examen microscopique\nIl montre des formations...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),ADENOCARCINOME MUCOSECRETANT - ADENOCARCINOME ...,ANATOMIE ET CYTOLOGIE PATHOLOGIQUES\n\nEXAMEN ...,tumorectomie
324,GSKCTUB10,P_ CARCINOME TUBULEUX,X,OHGSA7F0,Examen microscopique \nCe carcinome infiltrant...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),ADENOCARCINOME TUBULEUX (SAI),**SEIN DROIT : TUMORECTOMIE DU QUADRANT INFERO...,tumorectomie
325,GSKCYL10,P_K.CANAL ADENOIDE KYSTIQUE (CYLINDROME).,X,OHGSA7X6,Examen microscopique\nIl s'agit d'un carcinome...,PIECE OPERATOIRE AVEC EXERESE COMPLETE DE L'OR...,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),CYLINDROME MALIN (SAI) - CARCINOME ADENOIDE KY...,ZONECTOMIE - GANGLION SENTINELLE\n\nExamen mac...,tumorectomie
326,GSKF01,P_FICHE CANCER CONCLUSION,X,GSA7A0,Conclusion\n- Localisation : \n- Tumeur de typ...,None,SEIN (ÉGALEMENT UTILISÉ CHEZ L'HOMME),ADENOCARCINOME INVASIF (SAI),SEIN DROIT : EXERESE MAMMAIRE PARTIELLE (UNION...,tumorectomie


In [38]:
from __future__ import annotations

import json
import os
from typing import List
import unicodedata
from dotenv import load_dotenv
from openai import OpenAI
import re
from pydantic import BaseModel, Field, field_validator

load_dotenv()

MODEL_STEP1 = os.getenv("OPENAI_MODEL_STEP1", "gpt-5.4")
client = OpenAI()


class Step1Example(BaseModel):
    example_id: str
    report_text: str

class Step1Output(BaseModel):
    examples: List[Step1Example] = Field(min_length=1)


PROMPT_STEP1 = """
Tu es un médecin anatomopathologiste expert en pathologie mammaire.

Lis l’abrégé et le template brut ci-dessous et produis des exemples textuels instanciés.

OBJECTIF
Retourner EXHAUSTIVEMENT un exemple complet pour chaque combinaison médicalement possible des alternatives pertinentes présentes dans le template.

UTILISATION DE L’ABRÉGÉ
L’abrégé fournit un contexte diagnostique synthétique important pour interpréter correctement le template.
Utilise l’abrégé pour guider le type de lésion, le type de prélèvement, le caractère bénin ou malin, et pour construire un Titre et une Macroscopie cohérents.
En cas d’ambiguïté du template, privilégier une interprétation cohérente avec l’abrégé.


STRUCTURE OBLIGATOIRE
Chaque compte rendu final doit obligatoirement contenir :
- une section Titre
- une section Examen macroscopique ou Macroscopie

Si le template ne contient pas explicitement l’une de ces sections, tu dois l’ajouter avec un contenu bref, concret, neutre, cohérent et médicalement plausible, en restant au plus proche du template.
- Titre : intitulé médical simple et cohérent, avec si possible latéralité, type de prélèvement et localisation.
  Exemples: "SEIN GAUCHE : EXÉRÈSE D’UN NODULE DU QUADRANT SUPÉRO-EXTERNE", "SEIN DROIT : TUMORECTOMIE DU QSE AVEC GANGLION SENTINELLE", "MICROBIOPSIE MAMMAIRE 14G DU QSI DU SEIN GAUCHE"
- Macroscopie : description brève, plausible et cohérente avec le prélèvement, la lésion et la conclusion. 
  Elle peut comporter les dimensions, poids et aspect de la pièce, l’aspect de la lésion à la coupe, la distance à la berge la plus proche, les recoupes éventuelles et les ganglions éventuels.
  Exemple: "Pièce d’exérèse mammaire mesurant 3,5 x 2,8 x 2 cm et pesant 9 g.
  À la coupe, on individualise un nodule bien limité, ovalaire, blanchâtre, ferme, d’aspect fasciculé, mesurant 1,6 cm dans son plus grand axe.
  La lésion est située à 4 mm de la berge la plus proche.
  Le reste du parenchyme mammaire d’aspect macroscopique banal."
- Titre et Macroscopie: Ne décrire que les éléments cohérents avec le template.


ALTERNATIVES PERTINENTES À COMBINER
Alternatives pertinentes à prendre en compte seulement si elles sont réellement proposées comme options dans le template :
- type de prélèvement
- présence/absence de ganglions prélevés
- présence/absence de rupture capsulaire
- présence/absence d’emboles vasculaires
- statut RO/RP
- statut HER2
- marges saines ou atteintes
- Ki67 si c’est une vraie alternative textuelle
- grade SBR si c’est une vraie alternative textuelle

RÈGLES
- Identifier les alternatives même si elles sont exprimées de façon non standard ou implicite.
- Générer le produit combinatoire complet des alternatives pertinentes.
- Exclure toute combinaison médicalement impossible.
- Si aucune alternative pertinente n’existe, générer un seul exemple.
- Ne jamais inventer l’opposé d’une option absente du template.
- Une option isolée n’est pas une alternative binaire.
- Dans chaque exemple, garder une seule option par bloc alternatif et supprimer les autres.
- Les variations purement numériques ne créent pas de branche, mais doivent être instanciées par des valeurs plausibles et cohérentes.
- Le texte final doit rester aussi proche que possible du template.
- Ne pas inventer de données pronostiques majeures absentes du template.

BLOCS EXCLUSIFS
- Les conclusions RO/RP forment un bloc alternatif exclusif.
- Les conclusions HER2 forment un bloc alternatif exclusif.
- Dans chaque exemple, tout le texte détaillé doit être cohérent avec l’option RO/RP et HER2 retenue.

CONTRAINTES MÉDICALES
- Le grade SBR s’applique seulement aux carcinomes infiltrants.
- Ne pas inventer d’emboles vasculaires pour une lésion strictement in situ.
- ganglions_atteints <= ganglions_preleves
- si ganglions_atteints = 0, rupture_capsulaire = non
- toute instanciation doit être médicalement plausible et cohérente

AJOUTS EN CAS D’ABSENCE DANS LE TEMPLATE
- Si la section Titre est absente, l’ajouter avec un intitulé simple, neutre et cohérent avec le type de prélèvement.
- Si la section Macroscopie est absente, l’ajouter avec une formulation minimale descriptive, neutre, plausible et cohérente avec le prélèvement.
- Ne pas ajouter d’autres sections obligatoires non demandées.

SORTIE
Réponds uniquement avec le JSON conforme au schéma attendu.
Ne retourne jamais une liste vide.

ABRÉGÉ
{ABREGE}
TEMPLATE BRUT
{TEXT}
""".strip()


def build_step1_prompt(abrege: str,text: str) -> str:
    return PROMPT_STEP1.format(ABREGE=abrege,TEXT=text)


def step1_generate_examples(abrege:str, text: str) -> Step1Output:
    response = client.responses.parse(
        model=MODEL_STEP1,
        input=[
            {
                "role": "system",
                "content": (
                    "Tu es un anatomopathologiste rigoureux. "
                    "Tu génères des exemples complets à partir des alternatives textuelles pertinentes du template, "
                    "en respectant strictement les contraintes médicales et la structure demandée."
                ),
            },
            {
                "role": "user",
                "content": build_step1_prompt(abrege,text),
            },
        ],
        text_format=Step1Output,
    )
    return response.output_parsed


def run_step1(abrege:str, text: str) -> dict:
    result = step1_generate_examples(abrege,text)
    return result.model_dump(mode="json")




In [9]:
# do each 100 to see $$$
START=200
END=400
examples={}
examplesjson={}
for ind in df_completed_templates_sein.index:
    if ind >= START and ind < END: #ind >= 20 and ind <40: 
        print(ind)
        template_text=df_completed_templates_sein.iloc[ind].to_dict()["Text"]
        abrege=df_completed_templates_sein.iloc[ind].to_dict()["Abrégé"]
        out2 = run_step1(abrege,template_text)
        examples[ind]=out2
        print(out2)    

df = pd.DataFrame(examples)

# save dataframe
df.to_csv(f"last/examples_{END}.csv", index=False, encoding="utf-8")
df.to_pickle(f"last/examples_{END}.pkl")   # plus pratique pour recharger tel quel
# df = pd.read_pickle("examples.pkl")

# save json
with open(f"last/examples_{END}.json", "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)

In [1]:
import pandas as pd

files = [
    "last/examples_100.pkl",
    "last/examples_200.pkl",
    "last/examples_400.pkl",
]

dfs = []

for file in files:
    df = pd.read_pickle(file)

    # convert 1 row x N cols  ->  N rows x 1 col
    if df.shape[0] == 1:
        df = df.T.reset_index(drop=True)

    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)

output_path = "last/all_templates_examples.json"
full_df.to_json(output_path, orient="records", force_ascii=False, indent=2)

print(full_df.shape)
print(f"Saved to {output_path}")

(328, 1)
Saved to last/all_templates_examples.json


In [6]:
full_df.columns

Index(['examples'], dtype='object')

In [8]:
full_df.iloc[0].to_dict()

{'examples': [{'example_id': 'ex1',
   'report_text': 'Titre\nMICROBIOPSIE MAMMAIRE DU SEIN : LÉSION FIBRO-ÉPITHÉLIALE\n\nMacroscopie\nFragments de biopsie mammaire blanchâtres et jaunâtres, mesurant ensemble 1,4 cm dans leur plus grande dimension.\n\nExamen microscopique\nOn observe une lésion fibro-épithéliale constituée par une prolifération de galactophores de taille variée, étirés\net déformés par un tissu palléal fibrosé. Les sections canalaires sont tapissées par une double assise de cellules\nmyo-épithéliales et de cellules glandulaires régulières. Absence de lésion inflammatoire spécifique et de\nprolifération épithéliale atypique.\nEtude immuno-histochimique (procédure Dako Omnis Flex Module Omnis) :\nAE1-AE3 (GA053/ Dako Clone AE1/AE3) : absence de cellule atypique isolée dans la fibrose.\nP63 (GA662/Dako Clone DAK-p63) : positive sur les cellules myoepithéliales.\nKI67 (GA626/Dako Clone MIB-1) : <1%.\n\nConclusion\nAdénofibrome.\nAbsence de prolifération tumorale in situ ou